In [7]:
import pandas as pd
import numpy as np

# Revenue

In [15]:
rev = pd.read_csv('data/raw/cbdtp_revenues_expenses.csv')

rev

,Month,Category,Amount (in Millions)
0,2025-01-01,Program Expenses,-11.14
1,2025-01-01,Toll Revenue,48.66
2,2025-02-01,Program Expenses,-11.47
3,2025-02-01,Toll Revenue,51.92
4,2025-03-01,Program Expenses,-13.28
5,2025-03-01,Toll Revenue,58.43
6,2025-04-01,Program Expenses,-10.82
7,2025-04-01,Toll Revenue,56.73
8,2025-05-01,Program Expenses,-10.94
9,2025-05-01,Toll Revenue,61.04


In [21]:
# pivot the data to wide format
rev_wide = rev.pivot(index='Month', columns='Category', values='Amount (in Millions)').reset_index()

rev_wide['Program Net Revenue'] = rev_wide['Toll Revenue'] + rev_wide['Program Expenses']

rev_wide.to_csv('data/processed/revenue.csv', index=False)

# Survey Data

In [8]:
survey_data = pd.read_csv('data/raw/survey_data.csv')

In [9]:
survey_data['percentage'] = survey_data['percentage'] / 100

boroughs_survey = survey_data[(survey_data['category'] == 'Borough') | (survey_data['category'] == 'Overall')].copy()

boroughs_survey.drop(columns=['category'], inplace=True)
boroughs_survey.rename(columns={'subcategory': 'Borough'}, inplace=True)

boroughs_survey.to_csv('data/processed/survey_data.csv', index=False)

# Traffic Fatalities

In [10]:
tf = pd.read_csv('data/raw/traffic_fatalities.csv')

In [12]:
tf.columns

Index(['Year', 'Pedestrians', 'Traditional Bike', 'E-Bike', 'Moped',
       'Stand-up Scooter', 'Motorcycle', 'Off-Road', 'Other',
       'Motor Vehicle Occupants', 'Total'],
      dtype='object')

In [13]:
tf['Motorized Two-Wheelers'] = tf['Traditional Bike'] + tf['E-Bike'] + tf['Moped'] + tf['Stand-up Scooter'] + tf['Motorcycle'] + tf['Off-Road'] + tf['Other']

tf.drop(columns=['Traditional Bike', 'E-Bike', 'Moped', 'Stand-up Scooter', 'Motorcycle', 'Off-Road', 'Other', 'Total'], inplace=True)

tf.melt(id_vars=['Year'], var_name='Vehicle Type', value_name='Fatalities').to_csv('data/processed/traffic_fatalities.csv', index=False)

# MTA Crime

In [33]:
crime = pd.read_csv('data/raw/MTA_Major_Felonies_20260120.csv')

crime['Month'] = pd.to_datetime(crime['Month'], format='%m/%d/%Y')

crime = crime[crime['Agency'] == 'NYCT']

# replace null in the Crimes per Million Riders column with 0
crime['Crimes per Million Riders'] = crime['Crimes per Million Riders'].fillna(0)

crime.to_csv('data/processed/crime_data.csv', index=False)

# Performance

In [79]:
import requests

base_url = "https://data.ny.gov/resource/"

# Define dataset IDs and query
dataset_2025 = "nmu4-7tz9.json"
dataset_2020_2024 = "bg59-42xi.json"
query = "SELECT `month`, sum(`num_sched_trains`), sum(`num_actual_trains`) GROUP BY `month`"

# Fetch data with parameters
performance_2025_response = requests.get(base_url + dataset_2025, params={"$query": query})
performance_2020_2024_response = requests.get(base_url + dataset_2020_2024, params={"$query": query + " ORDER BY `month` DESC NULL FIRST"})

performance_2025_df = pd.DataFrame(performance_2025_response.json())
performance_2020_2024_df = pd.DataFrame(performance_2020_2024_response.json())

# concatenate the two dataframes
performance = pd.concat([performance_2020_2024_df, performance_2025_df])

performance_2025_df = pd.read_json(performance_2025)
performance_2020_2024_df = pd.read_json(performance_2020_2024)

# concatenate the two dataframes
performance = pd.concat([performance_2020_2024_df, performance_2025_df])

In [80]:
performance['month'] = pd.to_datetime(performance['month'], format='%Y-%m-%dT%H:%M:%S.%f')

In [81]:
performance['service_rate'] = performance['sum_num_actual_trains'] / performance['sum_num_sched_trains']

performance

,month,sum_num_sched_trains,sum_num_actual_trains,service_rate
0,2024-12-01,78274,73881,0.943877
1,2024-11-01,75197,71620,0.952432
2,2024-10-01,74253,71221,0.959167
3,2024-09-01,73604,70542,0.958399
4,2024-08-01,75778,71977,0.949840
...,...,...,...,...
6,2025-04-01,72751,70409,0.967808
7,2025-11-01,77181,73939,0.957995
8,2025-10-01,74823,71319,0.953169
9,2025-08-01,77151,73479,0.952405


In [82]:
# restrict to 2023 onwards
performance = performance[performance['month'] >= '2023-01-01'][['month', 'service_rate']]

performance.columns = ['Month', 'Service Rate (%)']

# Schedule Performance

In [83]:
schedule_2020_2024_dataset = "4apg-4kt9"
schedule_2025_dataset = "s4u6-t435"

query_schedule_2020_2024 = "SELECT `month`, median(`over_five_mins_perc`), median(`customer_journey_time`), median(`additional_platform_time`), median(`additional_train_time`) WHERE caseless_one_of(`period`, \"peak\") GROUP BY `month` HAVING `month` BETWEEN \"2023-01-01T00:00:00\" :: floating_timestamp AND \"2024-12-31T23:45:00\" :: floating_timestamp ORDER BY `month` DESC NULL FIRST"

query_schedule_2025 = "SELECT `month`, median(`additional_platform_time`), median(`additional_train_time`), median(`over_five_mins_perc`), median(`customer_journey_time`) WHERE caseless_one_of(`period`, \"peak\") GROUP BY `month`"


In [84]:
schedule_2020_2024_response = requests.get(base_url + schedule_2020_2024_dataset + ".json", params={"$query": query_schedule_2020_2024})
schedule_2025_response = requests.get(base_url + schedule_2025_dataset + ".json", params={"$query": query_schedule_2025})

schedule_2020_2024_df = pd.DataFrame(schedule_2020_2024_response.json())
schedule_2025_df = pd.DataFrame(schedule_2025_response.json())

schedule = pd.concat([schedule_2020_2024_df, schedule_2025_df])

schedule.columns = ['Month', 'Journey Time Performance (Over 5 Min %)', 'Journey Time Performance (Within 5 Min %)', 'Additional Platform Time (Minutes)', 'Additional Train Time (Minutes)']

schedule['Month'] = pd.to_datetime(schedule['Month'], format='%Y-%m-%dT%H:%M:%S.%f')

schedule

,Month,Journey Time Performance (Over 5 Min %),Journey Time Performance (Within 5 Min %),Additional Platform Time (Minutes),Additional Train Time (Minutes)
0,2024-12-01,0.137308385,0.86269162,1.1810536,0.438450285
1,2024-11-01,0.134132565,0.86586745,1.26439455,0.46452325
2,2024-10-01,0.1241476055,0.8758524,1.1092264,0.47343263
3,2024-09-01,0.130866065,0.8691339,1.18776805,0.4248323
4,2024-08-01,0.12557673,0.87442325,1.19958545,0.41236943
5,2024-07-01,0.126462595,0.87353743,1.2827806,0.27802735
6,2024-06-01,0.13791927,0.86208073,1.146675,0.35394326
7,2024-05-01,0.12763491,0.872365085,1.2261005,0.38236641
8,2024-04-01,0.12582326,0.874176735,1.15993565,0.40265644
9,2024-03-01,0.149107465,0.85089255,1.2224242,0.42958365


In [85]:
# merge performance and schedule data
performance = pd.merge(performance, schedule, on='Month')

performance.melt(id_vars=['Month'], var_name='Metric', value_name='Value').to_csv('data/processed/performance_data.csv', index=False)